# FlashEats — Class 8: Building a Dependable Pipeline
## From Exploratory Notebook to Dependable Production Data Foundation

### Core Engineering Question
> **“Our notebook worked once when we ran it cell-by-cell. Can it run every morning at 3:00 AM without the engineer who wrote it?”**

Today we transition from exploratory analysis to building a production data pipeline with:
- **Modular stages** (`extract`, `validate`, `clean`, `transform`, `save`, `log`)
- **Configuration outside code** (`.env` settings)
- **Bounded retries with backoff** for network calls
- **Strict validation gate** that blocks publishing if data contracts fail
- **Idempotent execution** (safe to rerun without duplicating data)
- **Controlled failure testing** (chaos mode)
- **Gate 2 Data Readiness Review** for executive decision-making


In [ ]:
import os, sys, subprocess, json, time
from pathlib import Path
import pandas as pd

# Verify working directory
BASE_DIR = Path.cwd()
print("Working directory:", BASE_DIR)
print("Pipeline script exists:", (BASE_DIR / "run_pipeline.py").exists())


---
## 1. Modular Pipeline Architecture

A dependable pipeline decouples responsibilities into distinct modules rather than keeping everything in one giant script:

```text
EXTRACT  ──▶  VALIDATE  ──▶  CLEAN  ──▶  TRANSFORM  ──▶  SAVE  ──▶  LOG
```

Let's inspect the files in our `pipeline/` package:


In [ ]:
for path in sorted((BASE_DIR / "pipeline").glob("*.py")):
    print(f"- {path.name:15s} ({path.stat().st_size:5d} bytes)")


### Engineering Reflection: Why not one giant function?
- **Diagnosability**: When a monolithic notebook or function crashes, you get an ambiguous 200-line stack trace. With modular stages, logs immediately identify whether the failure occurred during ingestion, validation, or transformation.
- **Fail-Fast Boundary**: The `save` stage should only ever execute after all validation checks have passed. Placing the persistence logic inside its own module ensures partially processed data is never exposed to downstream users.
- **Testability**: Each module (e.g. `gate.py` or `cleanse.py`) can be tested independently with unit tests.


---
## 2. Configuration Belongs Outside Core Code

Settings like API URLs, retry attempts, page sizes, and freshness thresholds change across development, staging, and production environments. Hardcoding them inside python scripts is an operational anti-pattern.

Let's inspect our pipeline configuration settings:


In [ ]:
print("--- config/.env.example ---")
print((BASE_DIR / "config" / ".env.example").read_text(encoding="utf-8"))


### Discussion: What belongs in configuration?
1. **Environment-specific endpoints**: In development, `DISPATCH_API_URL` points to `http://127.0.0.1:8000`. In production, it points to a cloud-managed service.
2. **Operational thresholds**: `MAX_RETRIES=3`, `RETRY_BASE_DELAY=1.0`, and `MAX_DATA_AGE_DAYS=60` are operational tuning parameters. They should be adjustable by on-call engineers without redeploying code.
3. **Secrets management**: API keys, database credentials, and service tokens must live in `.env` or a secrets manager, never hardcoded in source control.


---
## 3. Resilient Ingestion: Bounded Retries & Backoff

Network calls to external APIs will inevitably experience transient glitches.
- A **fragile pipeline** crashes on the first HTTP 500 error.
- A **reckless pipeline** retries in an infinite loop, flooding the upstream server.
- A **dependable pipeline** uses **bounded retries with exponential backoff**:
  - Transient errors (`HTTP 500` server errors, `HTTP 429` rate limits) are retried a limited number of times (e.g. 3 attempts) with increasing delays (1s, 2s, 4s).
  - Fatal errors (`HTTP 404` or missing schema columns) fail immediately without retrying because retrying a missing column will never fix it.


---
## 4. Single-Command Pipeline Execution

Let's execute the complete pipeline using a single CLI command for logical run date `2026-09-22`:


In [ ]:
# Run pipeline for 2026-09-22
cmd = [sys.executable, "run_pipeline.py", "--run-date", "2026-09-22"]
res = subprocess.run(cmd, capture_output=True, text=True)

print("Return Code:", res.returncode)
print("\n--- Pipeline Log Output ---")
for line in res.stdout.strip().split("\n"):
    print(line)
if res.stderr:
    print("\n--- Stderr ---")
    print(res.stderr)


Let's inspect the published outputs generated in the date-partitioned directory:


In [ ]:
output_dir = BASE_DIR / "data" / "processed" / "run_date=2026-09-22"
print("Generated output files:")
for f in sorted(output_dir.glob("*.*")):
    print(f" - {f.name:25s} ({f.stat().st_size} bytes)")

# Preview published metrics
metrics_path = output_dir / "metrics.json"
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
print("\nPublished Daily KPIs:")
print(json.dumps(metrics, indent=2))


---
## 5. Proving Idempotency (Rerun Safety)

A pipeline is **idempotent** if running it multiple times for the same input date produces the exact same result without duplicating records or corrupting files.

Let's test this by running the pipeline a second time for the exact same date:


In [ ]:
# Rerun the exact same pipeline command
rerun_res = subprocess.run(cmd, capture_output=True, text=True)
print("Rerun Exit Code:", rerun_res.returncode)

# Check row counts in order journey
df_first = pd.read_csv(output_dir / "order_journey.csv")
print(f"Total rows in published order journey: {len(df_first)}")
print(f"Unique order_ids: {df_first['order_id'].nunique()}")
print(f"Duplicate order_ids: {df_first['order_id'].duplicated().sum()}")


### Why did the rerun not duplicate data?
1. **Partition Isolation**: Outputs are written to `run_date=2026-09-22/`. Rerunning a day only touches that specific day's folder.
2. **Snapshot Overwrite**: The pipeline processes the full order state for that date, not an incremental append.
3. **Atomic File Writes**: The storage module writes to `.tmp` files first, then uses `os.replace` to swap them in one atomic filesystem operation. External readers never see a half-written file.


---
## 6. Controlled Failure: Chaos Testing

A dependable pipeline must not only succeed on good data—it must **safely and informatively fail on bad data**.

Let's test our Validation Gate against deliberate chaos defects:

### Test A: Schema Violation (`missing_column`)
What happens if an upstream database migration drops `promised_eta`?


In [ ]:
# Run with chaos mode: missing_column
chaos_cmd = [sys.executable, "run_pipeline.py", "--run-date", "2026-09-22", "--chaos", "missing_column"]
chaos_res = subprocess.run(chaos_cmd, capture_output=True, text=True)

print("Exit Code:", chaos_res.returncode)
print("\nLog lines on failure:")
for line in chaos_res.stdout.strip().split("\n")[-6:]:
    print(line)


**Notice**:
- The pipeline exited with **Code 2** (Validation Gate Halt).
- The error message specifically identified the broken contract: `orders: missing required columns: ['promised_eta']`.
- **Zero processed files were written.** The existing published files were not touched or overwritten with corrupt data.

---

### Test B: Data Freshness Violation (`stale_data`)
What happens if source data ingestion stalls and data is 400 days old?


In [ ]:
# Run with chaos mode: stale_data
stale_cmd = [sys.executable, "run_pipeline.py", "--run-date", "2026-09-22", "--chaos", "stale_data"]
stale_res = subprocess.run(stale_cmd, capture_output=True, text=True)

print("Exit Code:", stale_res.returncode)
for line in stale_res.stdout.strip().split("\n")[-6:]:
    print(line)


---
### Test C: Duplicate Handling (`duplicate_order`)
What happens if duplicate order records appear?


In [ ]:
# Run with chaos mode: duplicate_order
dupe_cmd = [sys.executable, "run_pipeline.py", "--run-date", "2026-09-22", "--chaos", "duplicate_order"]
dupe_res = subprocess.run(dupe_cmd, capture_output=True, text=True)

print("Exit Code:", dupe_res.returncode)
for line in dupe_res.stdout.strip().split("\n")[-8:]:
    print(line)


**Notice**:
- The validation gate flagged a `WARN` indicating duplicate rows.
- The cleaning stage applied the deduplication rule (keeping first record), dropping the duplicates and publishing exactly 1,600 clean orders.
- The pipeline exited with **Code 0**.


---
## 7. Gate 2: Data Readiness Review

Now that our pipeline is automated and dependable, we must answer the executive business question:

> **Is FlashEats data ready to train an AI delay prediction system?**

Let's inspect our Gate 2 review report:


In [ ]:
readiness_file = BASE_DIR / "GATE2_DATA_READINESS.md"
print(readiness_file.read_text(encoding="utf-8"))


### Summary of the Gate 2 Review:
- **Decision: NOT READY FOR AI MODELING**
- **Rationale**:
  1. **Disputed KPI Definition**: Stakeholders disagree on what "late" means (VP Ops: 56.4% vs. Support: 23.3%). An AI model cannot optimize for an undefined metric.
  2. **Missing Explanatory Feature**: The app does not track when drivers arrive at restaurants. Kitchen prep accounts for 0.83 correlation with delay, but we cannot tell if delays are caused by slow cooking or long driver waits outside the store.
  3. **Data Logging Gaps**: 37 delivered orders lack completion timestamps.
- **Path Forward**: Daily reporting pipeline can run today. The AI project should be paused until kitchen tablets and driver arrival geofences are instrumented.


---
## 8. Knowledge Check & Review

### Poll 1: The Dispatch API returns HTTP 500 once. What should the pipeline do?
- **Answer: B.** Retry a bounded number of times with exponential backoff. A single 500 error is usually a transient server hiccup that succeeds on the second try.

### Poll 2: `promised_eta` disappears from the source schema. What should happen?
- **Answer: C.** Halt execution immediately and fail the validation gate. Retrying will not restore a missing column, and continuing with NULLs would publish corrupted metrics.

### Poll 3: What makes a rerun idempotent?
- **Answer: C.** Running the same date replaces the target partition cleanly with the exact same row count and values, without duplicating rows or leaving orphaned temp files.

### Poll 4: What is the primary difference between a fragile pipeline and a dependable one?
- **Answer: A.** A fragile pipeline only works when everything is perfect and silently corrupts data on edge cases. A dependable pipeline has explicit validation firewalls, clear error logging, bounded retries, and guarantees atomic writes.
